# SCB Silver Layer, Workforce Data

Transforms raw SCB JSON from the bronze volume into clean, typed Delta tables.

**Source**: `labour_market_platform.bronze_work_scb.landing/scb` (external volume)  
**Target**: `labour_market_platform.silver_work_scb` (Delta tables)

| Silver Table | Source File(s) | Description |
|---|---|---|
| `wages` | `wages_*.json` | Wage distribution by sector, occupation, sex, year |
| `aku_employment` | `aku_employment_stock_occupation.json` | Employment stock by occupation, sex, year |
| `aku_population` | `aku_population_region_labourstatus.json` | Population by region, labour status, sex, year |
| `aku_unemployment` | `aku_unemployed_age_sex.json` | Unemployment by age group, sex, year |

**Transformations**: 
* Flatten nested JSON -> typed columns
* Standardize and alias column names
* Cast metrics to `DOUBLE`
* Handle suppressed values (`..`, `...`, `.`, etc.) as `NULL`
* Enforce `snake_case` naming conventions
* Deduplicate overlapping partitions (Wages)

In [0]:
import json, os, re
from glob import glob
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql import functions as F

# Define our Medallion Architecture paths
CATALOG       = "labour_market_platform"
BRONZE_SCHEMA = "bronze_work_scb"
SILVER_SCHEMA = "silver_work_scb"
VOLUME        = "landing"
BRONZE_ROOT   = "scb"
BASE_PATH     = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}/{BRONZE_ROOT}"

# Ensure our Silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

def get_latest_partition(subfolder: str) -> str:
    """Finds the most recent data drop in the Bronze layer."""
    parts = sorted(glob(f"{BASE_PATH}/{subfolder}/ingestion_date=*"))
    if not parts:
        raise FileNotFoundError(f"No partitions found under {BASE_PATH}/{subfolder}/")
    return parts[-1]

print(f"Bronze root:{BASE_PATH}")
print(f"Latest wages partition: {get_latest_partition('wages')}")
print(f"Latest AKU partition: {get_latest_partition('aku')}")

In [0]:
from pyspark.sql import DataFrame
def parse_scb_to_spark(file_paths: list, column_renames: dict = None) -> DataFrame:

    """
    Parses complex SCB JSON files into a flattened, typed PySpark DataFrame.
    """
    column_renames = column_renames or {}
    all_rows = []
    schema_fields = {} 

    # convert names to snake_case and generall expresion to remove signs and replace space
    def to_snake_case(s: str) -> str:
        return re.sub(r'[^a-z0-9]+', '_', s.lower().strip()).strip('_')

    # SCB uses these symbols for missing/suppressed data
    suppressed_vals = {'..', '...', '.', ''}

    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            raw = json.load(f)

        cols_meta = raw.get('columns', [])

        # Dynamically build the PySpark Schema
        for c in cols_meta:
            raw_name = to_snake_case(c['text'])
            final_name = column_renames.get(raw_name, raw_name)
            
            if final_name not in schema_fields:
                # Dimensions ('d') and Time ('t') are Strings. Metrics ('c') are Doubles.
                c_type = StringType() if c['type'] in ('d', 't') else DoubleType()
                schema_fields[final_name] = StructField(final_name, c_type, True)

        # Extract column names in order
        dim_names = [column_renames.get(to_snake_case(c['text']), to_snake_case(c['text'])) for c in cols_meta if c['type'] in ('d', 't')]
        val_names = [column_renames.get(to_snake_case(c['text']), to_snake_case(c['text'])) for c in cols_meta if c['type'] == 'c']

        # Flatten the nested data
        for row in raw.get('data', []):
            record = {}
            for name, val in zip(dim_names, row['key']):
                record[name] = val
            for name, val in zip(val_names, row['values']):
                # Standardize missing data to native SQL NULLs
                record[name] = None if val in suppressed_vals else float(val)
            all_rows.append(record)

    # Create a single PySpark DataFrame using the explicit schema
    unified_schema = StructType(list(schema_fields.values()))
    return spark.createDataFrame(all_rows, schema=unified_schema)

In [0]:
def save_to_silver(df, table_name: str, dedupe_columns: list):
    """
    Deduplicates the PySpark DataFrame and writes it to a Delta table.
    """
    # Deduplicate based on the primary keys
    df_clean = df.dropDuplicates(dedupe_columns)
    
    target_path = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    
    # Save as a Delta table
    df_clean.write.format("delta").mode("overwrite").saveAsTable(target_path)
    
    print(f"Successfully wrote {df_clean.count()} rows to {target_path}")

In [0]:
# PROCESS WAGES
WAGES_RENAMES = {
    "occupation_ssyk_2012": "occupation_code",
    
    # Base Salary Metrics
    "monthly_salary":   "monthly_salary_avg",
    
    # Percentiles adding salary and p as precentile suffix
    "10th_percentile":  "salary_p10",
    "25th_percentile":  "salary_p25",
    "75th_percentile":  "salary_p75",
    "90th_percentile":  "salary_p90",
    
    # Confidence Intervals 95% shortened to "ci95"
    "monthly_salary_95_percent_confidence_interval":    "monthly_salary_avg_ci95",
    "median_95_percent_confidence_interval":    "monthly_salary_median_ci95",
    "average_monthly_salary_10th_percentile_95_percent_confidence_interval":    "salary_p10_ci95",
    "average_monthly_salary_25th_percentile_95_percent_confidence_interval":    "salary_p25_ci95",
    "average_monthly_salary_75th_percentile_95_percent_confidence_interval":    "salary_p75_ci95",
    "average_monthly_salary_90th_percentile_95_percent_confidence_interval":    "salary_p90_ci95",
}

# Use glob to catch all wage files (e.g., wages_1.json, wages_2.json if split)
wages_files = glob(f"{get_latest_partition('wages')}/*.json")
df_wages = parse_scb_to_spark(wages_files, WAGES_RENAMES)

# Drop existing table to allow schema change (column renames updated)
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.wages")

# Save and deduplicate based on the grain of the data
save_to_silver(df_wages, "wages", dedupe_columns=["sector", "occupation_code", "sex", "year"])

In [0]:
# PROCESS AKU EMPLOYMENT
EMP_RENAMES = {
    "degree_of_attachment_to_the_labour_market": "attachment_code",
    "occupation":   "occupation_code",
    "employed_persons": "employed_thousands",
    "margin_of_error":  "moe_thousands",
}

# Grab the specific employment file
emp_files = glob(f"{get_latest_partition('aku')}/aku_employment_stock_occupation*.json")
df_emp = parse_scb_to_spark(emp_files, EMP_RENAMES)

# Save and deduplicate
save_to_silver(df_emp, "aku_employment", dedupe_columns=["attachment_code", "occupation_code", "sex", "year"])

In [0]:
# PROCESS AKU POPULATION
POP_RENAMES = {
    "labour_status":     "labour_status_code",
    "thousands":              "pop_thousands",
    "margin_of_error_1000s":  "moe_thousands",
    "percent":                  "pop_percent",
    "margin_of_error_percent":  "moe_percent",
}

# Grab the specific population file
pop_files = glob(f"{get_latest_partition('aku')}/aku_population_region_labourstatus*.json")
df_pop = parse_scb_to_spark(pop_files, POP_RENAMES)

# Save and deduplicate (grain is region, labor status, sex, and year)
save_to_silver(df_pop, "aku_population", dedupe_columns=["region", "labour_status_code", "sex", "year"])

In [0]:
# PROCESS AKU UNEMPLOYMENT
UNEMP_RENAMES = {
    "labour_status":        "labour_status_code",
    "age":                           "age_group",
    "1000s":                   "unemp_thousands",
    "margin_of_error_1000s":     "moe_thousands",
    "percent":                   "unemp_percent",
    "margin_of_error_percent":     "moe_percent",
}

# Grab the specific unemployment file
unemp_files = glob(f"{get_latest_partition('aku')}/aku_unemployed_age_sex*.json")
df_unemp = parse_scb_to_spark(unemp_files, UNEMP_RENAMES)

# Save and deduplicate (grain is labor status, age group, sex, and year)
save_to_silver(df_unemp, "aku_unemployment", dedupe_columns=["labour_status_code", "age_group", "sex", "year"])

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.wages LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_employment LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_population LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_unemployment LIMIT 10